In [2]:
import pandas as pd
import unicodedata

anvisa = pd.read_csv("tabela_anvisa_limpa.csv", sep=";", encoding="utf-8-sig")
drogasil = pd.read_csv("medicamento_mockados.csv")

drogasil["dataPreco"] = pd.to_datetime(drogasil["dataPreco"], errors="coerce")

In [3]:
def limpar_texto(t):
    if pd.isna(t):
        return ""
    t = str(t).lower().strip()
    # acentos
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    # facilitar o match
    t = t.replace("mg", " mg")
    t = t.replace("ml", " ml")
    t = t.replace("-", " ")
    t = " ".join(t.split())
    return t

In [4]:
# removendo as linhas sem substancias, se nao quebra o join
drogasil_sem_subst = drogasil[drogasil["substancia"].isna()].copy()
drogasil = drogasil[~drogasil["substancia"].isna()].copy()

print("Drogasil sem substância:", len(drogasil_sem_subst))
print("Drogasil com substância:", len(drogasil))

Drogasil sem substância: 27
Drogasil com substância: 2709


In [5]:
# limpeza e norm
#ANVISA 
anvisa["farmaco_limpo"] = anvisa["Fármaco"].apply(limpar_texto)

#DROGASIL
drogasil["substancia_limpa"] = drogasil["substancia"].apply(limpar_texto)

# lista de fármacos únicos da ANVISA pra usar no """like"""
farmacos = anvisa["farmaco_limpo"].dropna().unique().tolist()
print("Qtde fármacos únicos ANVISA:", len(farmacos))

Qtde fármacos únicos ANVISA: 191


In [6]:
# join com substr
# a cada subs limpa da drogasil
# retorna a lista de farmacos da anvisa que batem
def mapear_farmacos(sub):
    if not isinstance(sub, str) or not sub.strip():
        return []
    return [f for f in farmacos if f in sub]


#aplicando essa praga
# criando a lista que falei acima e conta quantos darmacos cada med tem
drogasil["farmacos_encontrados"] = drogasil["substancia_limpa"].apply(mapear_farmacos)
drogasil["qtde_farmacos"] = drogasil["farmacos_encontrados"].apply(len)

print("Distribuição de qtde de fármacos por medicamento:")
print(drogasil["qtde_farmacos"].value_counts())

Distribuição de qtde de fármacos por medicamento:
qtde_farmacos
1    2079
0     459
2     171
Name: count, dtype: int64


In [7]:

#separando oq tem e oq nao tem farmanco
drogasil["tem_farmaco"] = drogasil["qtde_farmacos"] > 0

# oq nao foi mapeado é natural
drogasil_ok = drogasil[drogasil["tem_farmaco"]].copy()
drogasil_sem_farmaco = drogasil[~drogasil["tem_farmaco"]].copy()

print("Com fármaco mapeado:", len(drogasil_ok))
print("Sem fármaco mapeado:", len(drogasil_sem_farmaco))

Com fármaco mapeado: 2250
Sem fármaco mapeado: 459


In [8]:
#expande os meds que tem mais de um farmaco
# se tem mais de um farmaco, mais de uma linha
drog_explo = drogasil_ok.explode("farmacos_encontrados")


# join da anvida com a drogasil
df_join = drog_explo.merge(
    anvisa,
    left_on="farmacos_encontrados",
    right_on="farmaco_limpo",
    how="left"
)
# linha da drogasil com linha da anvisa que corrsponde ao farmaco explodido
print("Shape do join many-to-many:", df_join.shape)

Shape do join many-to-many: (7353, 233)


In [9]:
colunas_excluir = [
    "Fármaco",
    "Forma farmacêutica",
    "Concentração máxima",
    "Indicação terapêutica simplificada",
    "indicacao_limpa",
    "indicacoes_list",
    "farmaco_limpo",
]


dummies_cols = [
    col for col in anvisa.columns
    if col not in colunas_excluir and anvisa[col].dtype != "O"
]

print("Qtd colunas de indicação/dummies:", len(dummies_cols))

Qtd colunas de indicação/dummies: 210


In [10]:
id_col = "idMedicamentos"

base_cols = [
    "nomeMedicamento",
    "Subgrupo terapêutico ou farmacológico",
    "preco",
    "marca",
    "quantidade",
    "dosagem",
    "substancia",
    "substancia_limpa",
    "avaliacao",          
    "dataPreco",          
    "idMedicamento_base",
]

agg_dict = {col: "first" for col in base_cols if col in df_join.columns}

# lista de fármacos distintos por medicamento
agg_dict["farmacos_encontrados"] = lambda x: sorted(
    set(f for f in x if isinstance(f, str))
)

# dummies: se qualquer fármaco tiver 1, o medicamento recebe 1
for col in dummies_cols:
    if col in df_join.columns:
        agg_dict[col] = "max"

df_final = df_join.groupby(id_col).agg(agg_dict).reset_index()

print("Shape df_final (1 linha por medicamento):", df_final.shape)
df_final.head()

Shape df_final (1 linha por medicamento): (2250, 223)


,idMedicamentos,nomeMedicamento,Subgrupo terapêutico ou farmacológico,preco,marca,quantidade,dosagem,substancia,substancia_limpa,avaliacao,...,sem_catarro,sem_catarro_associada_a_gripes_e_resfriados_ou_a_inalacao_de_agentes_irritantes,sob_os_seios_ou_em_outras_areas_da_pele_que_sofrem_atrito_em_criancas_e_adultos,teniase,tosse_e_dor_muscular_associadas_a_gripes_e_resfriados,tricuriase,urticarias,vulvar_e_peniana,vomito_e_distensao_abdominal,ulcera_cutanea
0,8,Pastilha Ciflogex Diet Sabor Menta 12 unidades,R02A – Preparações para a garganta,12.90,2,12un,3MG,Cloridrato de Benzidamina,cloridrato de benzidamina,4.8,...,0,0,0,0,0,0,0,0,0,0
1,9,Pastilha Ciflogex Sabor Mel e Limão 12 unidades,R02A – Preparações para a garganta,12.90,2,12un,3MG,Cloridrato de Benzidamina,cloridrato de benzidamina,4.5,...,0,0,0,0,0,0,0,0,0,0
2,10,Pastilha Ciflogex Sabor Menta e Limão 12 unidades,R02A – Preparações para a garganta,12.90,2,12un,3MG,Cloridrato de Benzidamina,cloridrato de benzidamina,3.2,...,0,0,0,0,0,0,0,0,0,0
3,11,Pastilha Flogoral Sabor Menta 4 unidades,R02A – Preparações para a garganta,6.45,3,4un,3MG,Cloridrato de Benzidamina,cloridrato de benzidamina,3.6,...,0,0,0,0,0,0,0,0,0,0
4,12,Pastilha Flogoral Sabor Menta 12 unidades,R02A – Preparações para a garganta,18.09,3,12un,3MG,Cloridrato de Benzidamina,cloridrato de benzidamina,3.2,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_final.to_csv("dataset_final_medicamentos.csv", index=False, encoding="utf-8-sig")


In [12]:
df_final.sample(10)[["nomeMedicamento", "substancia", "farmacos_encontrados"]]


,nomeMedicamento,substancia,farmacos_encontrados
773,Buprovil Ibuprofeno 300mg 30 comprimidos,Ibuprofeno,[ibuprofeno]
433,Resfenol 10 cápsulas,"Cloridrato de Fenilefrina,Paracetamol,Maleato ...",[paracetamol]
520,Alivium Ibuprofeno 400mg 3 cápsulas,Ibuprofeno,[ibuprofeno]
1632,"Maresis Baby 0,9% Descongestionante Spray Nasa...",Cloreto de Sódio,[cloreto de sodio]
872,Atroveran Dip Dipirona Monoidratada 1g 20 comp...,Dipirona Monoidratada,[dipirona]
2037,Dipirona Monoidratada 500mg 10 comprimidos EMS...,Dipirona,[dipirona]
1462,Mucofan Carbocisteína 20mg/ml Xarope Expectora...,Carbocisteína,[carbocisteina]
1780,Dipirona Monoidratada 500mg 10 comprimidos Med...,Dipirona,[dipirona]
468,Vibral 3mg/ml Sabor Framboesa Xarope 120ml,Dropropizina,[dropropizina]
876,Buscofem Ibuprofeno 400mg 2 cápsulas,Ibuprofeno,[ibuprofeno]
